In [1]:
print("rrkk")

rrkk


In [2]:
!nvidia-smi


Sat Oct 18 10:15:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121


In [4]:
!pip install ultralytics --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 79.0 MB/s eta 0:00:00


In [5]:
!pip install kaggle
from google.colab import files
files.upload()  # upload kaggle.json from your computer


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"kriisanfarmer","key":"c5de468e82fbb228538af695f16732fd"}'}

In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [7]:
!kaggle datasets download -d kriisanfarmer/guaallyolov11
!unzip guaallyolov11.zip -d /content/dataset


Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.2e9940bc77ead7a7d9fd9108c985287a_dup683.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.5d23cdd8168771810485139c8f697c5d.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.5d23cdd8168771810485139c8f697c5d_dup256.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.5d23cdd8168771810485139c8f697c5d_dup80.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.fe80c6bf636c5c1136e6380eab6d29d6.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.fe80c6bf636c5c1136e6380eab6d29d6_dup359.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.fe80c6bf636c5c1136e6380eab6d29d6_dup439.txt  
  inflating: /content/dataset/yolov11all/train/labels/scab0040ba_jpg.rf.fe80c6bf636c5c1136e6380eab6d29d6_dup519.txt  
  inflating: /conten

In [8]:
yaml_text = """
train: /content/dataset/data/train
val: /content/dataset/data/valid
test: /content/dataset/data/test

nc: 9
names:
  - algal_leaf_spot
  - black_mold
  - fruit_healthy
  - healthy_leaf
  - insect_bite
  - red_rust
  - scab
  - scorch
  - yellow_leaf_disease

bbox_type: obb
"""
open("/content/dataset/leaf_obb.yaml", "w").write(yaml_text)
print("✅ Wrote /content/dataset/leaf_obb.yaml")


✅ Wrote /content/dataset/leaf_obb.yaml


In [11]:
!ls -R /content/dataset


Streaming output truncated to the last 5000 lines.
scab0040ba_jpg.rf.fe80c6bf636c5c1136e6380eab6d29d6.txt
scab0041ba_jpg.rf.39522e3704de57924470dd6ed86692dc_dup412.txt
scab0041ba_jpg.rf.39522e3704de57924470dd6ed86692dc_dup663.txt
scab0041ba_jpg.rf.39522e3704de57924470dd6ed86692dc.txt
scab0041ba_jpg.rf.4a13f34ca755d6bb9cfde83cc6dcc0cb_dup734.txt
scab0041ba_jpg.rf.4a13f34ca755d6bb9cfde83cc6dcc0cb.txt
scab0042ba_jpg.rf.75e2751d85d0de306758892f4b20ea1e_dup285.txt
scab0042ba_jpg.rf.75e2751d85d0de306758892f4b20ea1e.txt
scab0042ba_jpg.rf.88b5f814544c47012f048ebb8fdc03a5_dup169.txt
scab0042ba_jpg.rf.88b5f814544c47012f048ebb8fdc03a5.txt
scab0043ba_jpg.rf.1a317c19c20c5bc90ec2b44e8dbdbe90_dup311.txt
scab0043ba_jpg.rf.1a317c19c20c5bc90ec2b44e8dbdbe90.txt
scab0043ba_jpg.rf.4c69da24b88dd18c7ef9243a4c1af7c4.txt
scab0043ba_jpg.rf.683011bdec1c5f3795dc44dd010a4013.txt
scab0044ba_jpg.rf.5ce4db419d3db4f364fe0e41b196fad7.txt
scab0044ba_jpg.rf.a1d7336c1de3bfedf10f4ab7677bc592.txt
scab0045ba_jpg.rf.5dbe55819

In [12]:
!find /content/dataset -type d


/content/dataset
/content/dataset/yolov11all
/content/dataset/yolov11all/test
/content/dataset/yolov11all/test/images
/content/dataset/yolov11all/test/labels
/content/dataset/yolov11all/train
/content/dataset/yolov11all/train/images
/content/dataset/yolov11all/train/labels
/content/dataset/yolov11all/valid
/content/dataset/yolov11all/valid/images
/content/dataset/yolov11all/valid/labels


In [13]:
yaml_text = """
path: /content/dataset/yolov11all

train: train/images
val: valid/images
test: test/images

nc: 9
names:
  - algal_leaf_spot
  - black_mold
  - fruit_healthy
  - healthy_leaf
  - insect_bite
  - red_rust
  - scab
  - scorch
  - yellow_leaf_disease

bbox_type: obb
"""
open("/content/dataset/leaf_obb.yaml", "w").write(yaml_text)
print("✅ Wrote fixed /content/dataset/leaf_obb.yaml")


✅ Wrote fixed /content/dataset/leaf_obb.yaml


In [14]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="/content/dataset/leaf_obb.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="leaf_yolo11_obb_training",
    name="yolo11n_obb_leaf",
    device=0       # ✅ now correctly uses the T4 GPU
)


Ultralytics 8.3.217 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/leaf_obb.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n_obb_leaf3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=T

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c0d69acc5c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0470

In [16]:
# =========================================
# Download all YOLOv11n training results
# =========================================
import shutil
from google.colab import files
import os

# Path to your training run folder
run_folder = "/content/leaf_yolo11_obb_training/yolo11n_obb_leaf3"
zip_name = "yolo11n_obb_all_results.zip"
zip_path = os.path.join("/content", zip_name)

if os.path.exists(run_folder):
    # Remove previous zip if it exists
    if os.path.exists(zip_path):
        os.remove(zip_path)

    # Create zip of the entire training folder
    shutil.make_archive(base_name=zip_path.replace(".zip",""), format='zip', root_dir=run_folder)
    print(f"📦 ZIP created successfully: {zip_path}")

    # Trigger download in Colab
    files.download(zip_path)
else:
    print(f"❌ Run folder not found: {run_folder}")


📦 ZIP created successfully: /content/yolo11n_obb_all_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>